# Przywracanie interpunkcji — BiLSTM: PolEval i Polish News

**Projekt z Inżynierii Lingwistycznej**

Notatnik uczy BiLSTM przewidywania znaku interpunkcyjnego po każdym słowie. Łączymy teksty z `expected.tsv` i kolumny `content` pliku `polish-news.csv` w jeden zbiór. Oba źródła przechodzą tę samą tokenizację i wyznaczanie etykiet. Następnie dzielimy dokumenty na trening (80%), walidację (10%) i test (10%). Klasy to `BRAK`, `.`, `,`, `?`, `!` i `:`.

## 1. Przygotowanie danych

Ustaw `DATA_DIR` na katalog z `expected.tsv`. Plik `polish-news.csv` umieść obok notatnika lub zmień `NEWS_PATH`. CSV musi mieć nagłówek `content`; inne kolumny są pomijane.

- **wejście `X`** — słowa bez interpunkcji,
- **etykiety `y`** — znak po każdym słowie w oryginalnym tekście.

Jeden wiersz `expected.tsv` lub jeden rekord CSV stanowi dokument. Rekord CSV może zawierać tekst wielowierszowy ujęty w cudzysłowy. Pomijamy puste teksty i teksty bez słów. Najpierw łączymy oba źródła, potem losowo dzielimy całe dokumenty. Słownik powstaje wyłącznie na zbiorze treningowym.

In [1]:
from pathlib import Path
import json
import random
import re
from collections import Counter

import numpy as np
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urządzenie:", DEVICE)


Urządzenie: cuda


## 2. Etykiety i wczytanie tekstów

Po każdym słowie model przewiduje jedną z sześciu klas. Funkcja `punctuation_in_gap` analizuje fragment między końcem słowa a początkiem następnego. Myślnik nie jest osobną klasą. Wspólna funkcja `make_document` przetwarza zarówno teksty PolEval, jak i wiadomości z kolumny `content`.

In [2]:
LABELS = ["BRAK", ".", ",", "?", "!", ":"]
label2id = {label: i for i, label in enumerate(LABELS)}
id2label = {i: label for label, i in label2id.items()}

DATA_DIR = Path("data/poleval/train")  # katalog z expected.tsv
if not (DATA_DIR / "expected.tsv").exists():
    for candidate in (Path("2021-punctuation-restoration-main/train"), Path(".")):
        if (candidate / "expected.tsv").exists():
            DATA_DIR = candidate
            break

DELIMITER = ","  # zmień na ";", jeśli plik używa średników

import csv
from itertools import islice

NEWS_PATH = Path("polish-news.csv")
NEWS_LIMIT = 10000  # None = wszystkie rekordy CSV

EXCLAMATION_PATH = Path("zdania_polaczone_4000.csv")
EXCLAMATION_LIMIT = None  # None = wszystkie rekordy CSV

# Długie artykuły mogą przekraczać domyślny limit pola CSV.
csv.field_size_limit(100_000_000)

def read_tsv(path):
    with path.open(encoding="utf-8-sig", newline="") as f:
        return [row for row in csv.reader(f, delimiter="\t") if row]

def punctuation_in_gap(gap):
    for mark in ["?", "!", ".", ",", ":"]:
        if mark in gap:
            return mark
    return "BRAK"

def make_document(doc_id, expected_text):
    # Te same wyodrębnione słowa tworzą wejście i etykiety.
    matches = list(re.finditer(r"\w+", expected_text, flags=re.UNICODE))
    tokens = [match.group(0) for match in matches]
    labels = []
    for i, match in enumerate(matches):
        gap_end = matches[i + 1].start() if i + 1 < len(matches) else len(expected_text)
        gap = expected_text[match.end():gap_end]
        labels.append(label2id[punctuation_in_gap(gap)])
    return {"id": doc_id, "tokens": tokens, "labels": labels}

expected_path = DATA_DIR / "expected.tsv"
if not expected_path.exists():
    raise FileNotFoundError(f"Nie znaleziono {expected_path}. Ustaw DATA_DIR na katalog zawierający expected.tsv.")

if not NEWS_PATH.exists():
    raise FileNotFoundError(f"Nie znaleziono {NEWS_PATH}. Ustaw NEWS_PATH na plik polish-news.csv.")

poleval_documents = []
for row_number, row in enumerate(read_tsv(expected_path), start=1):
    doc = make_document(f"poleval_{row_number}", " ".join(row).strip())
    if doc["tokens"]:
        doc["source"] = "poleval"
        poleval_documents.append(doc)

news_documents = []
with NEWS_PATH.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter=DELIMITER)
    if not reader.fieldnames or "content" not in reader.fieldnames:
        raise ValueError("CSV musi zawierać kolumnę 'content'. Sprawdź nagłówek i NEWS_DELIMITER.")
    for row_number, row in enumerate(islice(reader, NEWS_LIMIT), start=1):
        content = (row.get("content") or "").strip()
        doc = make_document(f"news_{row_number}", content)
        if doc["tokens"]:
            doc["source"] = "polish-news"
            news_documents.append(doc)


exclamation_documents = []
with EXCLAMATION_PATH.open(encoding="utf-8-sig", newline="") as f:
    reader = csv.DictReader(f, delimiter=DELIMITER)
    if not reader.fieldnames or "content" not in reader.fieldnames:
        raise ValueError("CSV musi zawierać kolumnę 'content'. Sprawdź nagłówek i EXCLAMATION_DELIMITER.")
    for row_number, row in enumerate(islice(reader, EXCLAMATION_LIMIT), start=1):
        content = (row.get("content") or "").strip()
        doc = make_document(f"exclamation_{row_number}", content)
        if doc["tokens"]:
            doc["source"] = "exclamation"
            exclamation_documents.append(doc)

# Jeden wspólny zbiór, przed podziałem na train/validation/test.
documents = poleval_documents + news_documents + exclamation_documents

print(f"Dokumenty PolEval: {len(poleval_documents)}")
print(f"Wiadomości Polish News: {len(news_documents)}")
print(f"Dokumenty Wykrzyknik: {len(exclamation_documents)}")
print(f"Łącznie dokumentów: {len(documents)}")
print(f"Łącznie słów: {sum(len(doc['tokens']) for doc in documents):,}")
print("Przykładowy tekst wejściowy:", " ".join(documents[0]["tokens"][:12]))
print("Etykiety:", [id2label[i] for i in documents[0]["labels"][:12]])

Dokumenty PolEval: 800
Wiadomości Polish News: 9988
Dokumenty Wykrzyknik: 4000
Łącznie dokumentów: 14788
Łącznie słów: 3,246,633
Przykładowy tekst wejściowy: w wywiadzie dla polski jarosław kaczyński podkreślił że informacje dotyczące radosława sikorskiego
Etykiety: ['BRAK', 'BRAK', 'BRAK', 'BRAK', 'BRAK', 'BRAK', ',', 'BRAK', 'BRAK', 'BRAK', 'BRAK', 'BRAK']


## 3. Szybki przegląd danych

Zliczamy etykiety w połączonym zbiorze obu źródeł. Klasy są niezrównoważone: brak znaku i przecinek mogą występować dużo częściej niż np. pytajnik. Dlatego oprócz accuracy będziemy patrzeć na F1 dla każdej klasy.

In [3]:
label_counts = Counter(label for doc in documents for label in doc["labels"])
for i, label in enumerate(LABELS):
    print(f"{label:>5}: {label_counts.get(i, 0):>10,}")

print("Przykładowe słowo i etykieta:")
for token, y in zip(documents[1]["tokens"][:15], documents[1]["labels"][:15]):
    print(f"{token:16s} -> {id2label[y]}")


 BRAK:  2,849,034
    .:    194,888
    ,:    187,137
    ?:      4,513
    !:      2,489
    ::      8,572
Przykładowe słowo i etykieta:
dowody           -> BRAK
prokuratury      -> BRAK
przedstawiające  -> ,
że               -> BRAK
janusz           -> BRAK
kaczmarek        -> ,
były             -> BRAK
szef             -> BRAK
mswia            -> ,
kłamał           -> ,
wypierając       -> BRAK
się              -> BRAK
spotkania        -> BRAK
z                -> BRAK
ryszardem        -> BRAK


## 4. Podział dokumentów i słownik

Połączone dane z `expected.tsv` i `polish-news.csv` dzielimy losowo na **80% treningu, 10% walidacji i 10% testu** (z zaokrągleniem liczebności). Podział odbywa się na poziomie całych dokumentów, ze stałym ziarnem `SEED`. Oba źródła uczestniczą we wspólnym losowaniu; nie wymuszamy proporcji źródeł w każdym zbiorze. Słownik budujemy wyłącznie z `train_docs`, aby nie korzystać ze słów walidacji i testu podczas jego tworzenia.

In [4]:
train_docs, temp_docs = train_test_split(documents, test_size=0.2, random_state=SEED)
valid_docs, test_docs = train_test_split(temp_docs, test_size=0.5, random_state=SEED)

for name, docs in (("Trening", train_docs), ("Walidacja", valid_docs), ("Test", test_docs)):
    sources = Counter(doc["source"] for doc in docs)
    print(f"{name}: {len(docs)} dokumentów; źródła: {dict(sources)}")

PAD, UNK = "<PAD>", "<UNK>"
word_counts = Counter(token for doc in train_docs for token in doc["tokens"])
vocab = {PAD: 0, UNK: 1}
for word, _ in word_counts.most_common():
    vocab[word] = len(vocab)
print("Rozmiar słownika:", len(vocab))

Trening: 11830 dokumentów; źródła: {'polish-news': 7977, 'exclamation': 3224, 'poleval': 629}
Walidacja: 1479 dokumentów; źródła: {'polish-news': 1001, 'exclamation': 387, 'poleval': 91}
Test: 1479 dokumentów; źródła: {'exclamation': 389, 'polish-news': 1010, 'poleval': 80}
Rozmiar słownika: 120103


## 5. Przygotowanie batchy

Dokumenty mają różną długość, więc w batchu dopełniamy krótsze sekwencje tokenem `<PAD>`. Etykiety dopełnienia dostają wartość `-100`, którą PyTorch pomija przy liczeniu straty.


In [5]:
class PunctuationDataset(Dataset):
    def __init__(self, docs, vocab):
        self.docs, self.vocab = docs, vocab
    def __len__(self):
        return len(self.docs)
    def __getitem__(self, idx):
        doc = self.docs[idx]
        x = [self.vocab.get(w, self.vocab[UNK]) for w in doc["tokens"]]
        y = doc["labels"]
        return torch.tensor(x, dtype=torch.long), torch.tensor(y, dtype=torch.long)

def collate_batch(batch):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs], dtype=torch.long)
    x_pad = pad_sequence(xs, batch_first=True, padding_value=0)
    y_pad = pad_sequence(ys, batch_first=True, padding_value=-100)
    return x_pad, y_pad, lengths

BATCH_SIZE = 16
train_loader = DataLoader(PunctuationDataset(train_docs, vocab), batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
valid_loader = DataLoader(PunctuationDataset(valid_docs, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(PunctuationDataset(test_docs, vocab), batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)


## 6. Model BiLSTM

Embedding zamienia indeks słowa na wektor. Dwukierunkowy LSTM analizuje kontekst przed i po słowie. Warstwa liniowa zwraca `len(LABELS)` wyników — po jednym dla każdej obsługiwanej klasy.

In [6]:
class BiLSTMPunctuationTagger(nn.Module):
    def __init__(self, vocab_size, embedding_dim=128, hidden_dim=128, num_classes=len(LABELS), dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.dropout = nn.Dropout(dropout)
        self.lstm = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=2,
            batch_first=True,
            bidirectional=True,
            dropout=0.2,
        )
        self.classifier = nn.Linear(hidden_dim * 2, num_classes)
    def forward(self, x, lengths):
        embedded = self.dropout(self.embedding(x))
        packed = pack_padded_sequence(embedded, lengths.cpu(), batch_first=True, enforce_sorted=False)
        encoded, _ = self.lstm(packed)
        encoded, _ = pad_packed_sequence(encoded, batch_first=True, total_length=x.size(1))
        return self.classifier(self.dropout(encoded))

model = BiLSTMPunctuationTagger(len(vocab), num_classes=len(LABELS)).to(DEVICE)
print(model)

BiLSTMPunctuationTagger(
  (embedding): Embedding(120103, 128, padding_idx=0)
  (dropout): Dropout(p=0.3, inplace=False)
  (lstm): LSTM(128, 128, num_layers=2, batch_first=True, dropout=0.2, bidirectional=True)
  (classifier): Linear(in_features=256, out_features=6, bias=True)
)


## 7. Trening

Używamy ważonej straty, żeby rzadkie klasy miały większy wpływ na uczenie. Wagi wyliczamy z częstości w zbiorze treningowym. To prosta metoda; można później porównać ją z treningiem bez wag.


In [7]:
import copy

# Wagi klas pomagają rzadziej występującym etykietom
# mieć większy wpływ na uczenie.
counts = np.array(
    [
        sum(label == i for doc in train_docs for label in doc["labels"])
        for i in range(len(LABELS))
    ],
    dtype=np.float32,
)

weights = 1.0 / np.sqrt(np.maximum(counts, 1.0))
weights = weights / weights.mean()
class_weights = torch.tensor(
    weights,
    dtype=torch.float32,
    device=DEVICE,
)

criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    ignore_index=-100,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)


def run_epoch(loader, training):
    model.train(training)
    total_loss = 0.0
    batches = 0

    for x, y, lengths in loader:
        x = x.to(DEVICE)
        y = y.to(DEVICE)

        with torch.set_grad_enabled(training):
            logits = model(x, lengths)
            loss = criterion( 
                logits.reshape(-1, len(LABELS)),
                y.reshape(-1),
            ) # Obliczenie straty dla bieżącego batcha.

            if training:
                optimizer.zero_grad() # Zerowanie gradientów przed obliczeniem nowych.
                loss.backward() # Obliczenie gradientów dla wszystkich parametrów modelu.
                nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Ograniczenie normy gradientu do 1.0, aby zapobiec eksplozji gradientów.
                optimizer.step() # Aktualizacja parametrów modelu na podstawie obliczonych gradientów.

        total_loss += loss.item()
        batches += 1

    return total_loss / max(batches, 1)


# Early stopping monitoruje stratę na zbiorze walidacyjnym.
EPOCHS = 50
EARLY_STOPPING_PATIENCE = 3
MIN_DELTA = 1e-4

best_val_loss = float("inf")
best_epoch = 0
epochs_without_improvement = 0
best_model_state = copy.deepcopy(model.state_dict())

for epoch in range(1, EPOCHS + 1):
    train_loss = run_epoch(train_loader, training=True)
    valid_loss = run_epoch(valid_loader, training=False)

    print(
        f"Epoka {epoch:02d} | "
        f"train loss: {train_loss:.4f} | "
        f"val loss: {valid_loss:.4f}"
    )

    if valid_loss < best_val_loss - MIN_DELTA:
        best_val_loss = valid_loss
        best_epoch = epoch
        epochs_without_improvement = 0
        best_model_state = copy.deepcopy(model.state_dict())
    else:
        epochs_without_improvement += 1
        print(
            "Brak poprawy walidacyjnej: "
            f"{epochs_without_improvement}/{EARLY_STOPPING_PATIENCE}"
        )

        if epochs_without_improvement >= EARLY_STOPPING_PATIENCE:
            print("Early stopping — zatrzymuję trening.")
            break

# Przywracamy wagi z epoki z najlepszym wynikiem walidacyjnym.
model.load_state_dict(best_model_state)
print(
    f"Przywrócono wagi z epoki {best_epoch} "
    f"(najlepszy val loss: {best_val_loss:.4f})."
)

Epoka 01 | train loss: 1.2873 | val loss: 1.0774
Epoka 02 | train loss: 1.0461 | val loss: 0.9909
Epoka 03 | train loss: 0.9702 | val loss: 0.9109
Epoka 04 | train loss: 0.9101 | val loss: 0.8543
Epoka 05 | train loss: 0.8631 | val loss: 0.8112
Epoka 06 | train loss: 0.8234 | val loss: 0.7741
Epoka 07 | train loss: 0.7881 | val loss: 0.7359
Epoka 08 | train loss: 0.7588 | val loss: 0.7049
Epoka 09 | train loss: 0.7320 | val loss: 0.6878
Epoka 10 | train loss: 0.7102 | val loss: 0.6637
Epoka 11 | train loss: 0.6921 | val loss: 0.6476
Epoka 12 | train loss: 0.6769 | val loss: 0.6317
Epoka 13 | train loss: 0.6617 | val loss: 0.6175
Epoka 14 | train loss: 0.6520 | val loss: 0.6036
Epoka 15 | train loss: 0.6402 | val loss: 0.5930
Epoka 16 | train loss: 0.6265 | val loss: 0.5817
Epoka 17 | train loss: 0.6164 | val loss: 0.5724
Epoka 18 | train loss: 0.6068 | val loss: 0.5661
Epoka 19 | train loss: 0.5978 | val loss: 0.5593
Epoka 20 | train loss: 0.5891 | val loss: 0.5468
Epoka 21 | train los

## 8. Ocena na zbiorze testowym

Raport pokazuje precision, recall i F1 osobno dla każdej klasy. Test pochodzi z połączonych danych PolEval i Polish News odłożonych przed treningiem; nie jest to oficjalny test PolEval. Do strojenia modelu i early stopping używamy wyłącznie walidacji.

In [8]:
def predict_loader(loader):
    model.eval()
    all_true, all_pred = [], []
    with torch.no_grad():
        for x, y, lengths in loader:
            logits = model(x.to(DEVICE), lengths)
            pred = logits.argmax(dim=-1).cpu()
            for i, length in enumerate(lengths.tolist()):
                all_true.extend(y[i, :length].tolist())
                all_pred.extend(pred[i, :length].tolist())
    return all_true, all_pred

y_true, y_pred = predict_loader(test_loader)
all_labels = list(range(len(LABELS)))
print(classification_report(
    y_true, y_pred, labels=all_labels, target_names=LABELS, zero_division=0, digits=3
))
print("Macro F1:", round(f1_score(y_true, y_pred, labels=all_labels, average="macro", zero_division=0), 4))
print("Weighted F1:", round(f1_score(y_true, y_pred, labels=all_labels, average="weighted", zero_division=0), 4))


              precision    recall  f1-score   support

        BRAK      0.965     0.963     0.964    293544
           .      0.763     0.787     0.775     19901
           ,      0.670     0.663     0.666     18849
           ?      0.513     0.531     0.521       458
           !      0.838     0.821     0.829       240
           :      0.379     0.443     0.408       899

    accuracy                          0.933    333891
   macro avg      0.688     0.701     0.694    333891
weighted avg      0.934     0.933     0.934    333891

Macro F1: 0.6941
Weighted F1: 0.9335


## 9. Predykcja dla własnego zdania

Podaj zdanie bez interpunkcji. Funkcja wyodrębni z niego słowa tak samo jak przy przygotowaniu treningu, a następnie wypisze przewidywany znak po każdym słowie. Nie przewidujemy myślnika.


In [9]:
def predict_sentence(sentence):
    tokens = [m.group(0) for m in re.finditer(r"\w+", sentence, flags=re.UNICODE)]
    if not tokens:
        return []
    ids = [vocab.get(token, vocab[UNK]) for token in tokens]
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    lengths = torch.tensor([len(ids)], dtype=torch.long)
    model.eval()
    with torch.no_grad():
        pred = model(x, lengths).argmax(dim=-1)[0].cpu().tolist()
    return list(zip(tokens, [id2label[i] for i in pred]))

def format_prediction(predictions):
    out = ""
    for token, label in predictions:
        mark = "" if label == "BRAK" else label
        out += token + mark + " "
    return out.strip()

example = "Kiedy wrócisz daj mi znać"
predictions = predict_sentence(example)
print(predictions)
print(format_prediction(predictions))


[('Kiedy', 'BRAK'), ('wrócisz', 'BRAK'), ('daj', 'BRAK'), ('mi', 'BRAK'), ('znać', '?')]
Kiedy wrócisz daj mi znać?


## 10. Zapis modelu

In [10]:
from pathlib import Path
import torch

Path("models/bilstm").mkdir(parents=True, exist_ok=True)

torch.save(
    {
        "state_dict": model.state_dict(),
        "vocab": vocab,
        "labels": LABELS,
        "config": {
            "vocab_size": model.embedding.num_embeddings,
            "embedding_dim": model.embedding.embedding_dim,
            "hidden_dim": model.lstm.hidden_size,
            "num_classes": model.classifier.out_features,
            "dropout": model.dropout.p,
        },
    },
    "models/bilstm/checkpoint.pt",
)